# 04 — İkili model

En iyi Kaggle skoru **sub_04 = 1.06168** (`yilin_gunu` yok, `leaves=31`).

Test satırlarının %22'si train'de hiç görünmeyen trafolar. Bu satırlarda
trafo geçmişi yok. Tek model hem geçmişi olan hem olmayan trafoları
birlikte öğrenmeye çalışıyor.

İkili model bunu ayırıyor:

- geçmişi olan satırlar → sub_04 tahmini (aynı kalır)
- yeni trafolar → sadece güç, lokasyon ve takvim kullanan ayrı model

Test anında trafonun train'de olup olmadığını biliyoruz, bu yüzden
yönlendirme doğru.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])


def rmsle(gercek, tahmin):
    return np.sqrt(np.mean((np.log1p(tahmin) - np.log1p(gercek))**2))


# 444 satır güç x 24 saat sınırını aşıyor, 23 trafoda toplanmış.
# Global kesmek yerine o trafonun medyanı ile düzeltiyoruz.
train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    medyan = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = medyan

TATILLER = pd.to_datetime([
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23',
    '2025-05-01','2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09',
    '2025-08-30','2025-10-29',
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]


def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il']    = lok[0]
    df['bolge'] = lok[1]
    df['ilce']  = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')

    df['ay']            = df['tarih'].dt.month
    df['gun']           = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu']    = (df['haftanin_gunu'] >= 5).astype(int)
    df['tatil']         = df['tarih'].dt.date.isin(TATILLER).astype(int)

    df['guc_bucket']     = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    return df


train = temel_ozellikler(train)
test  = temel_ozellikler(test)

KATEGORIK = ['il', 'bolge', 'ilce', 'guc_bucket_str']

print('train', train.shape, '| test', test.shape)
print('düzeltilen satır:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum())


train (1226237, 17) | test (714688, 15)
düzeltilen satır: 444


In [ ]:
def trafo_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])

    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis', 'mean'),
        trafo_medyan     = ('tuketim_duzeltilmis', 'median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis', 'count'),
        log_ortalama     = ('log_t', 'mean'),
        log_medyan       = ('log_t', 'median'),
        log_std          = ('log_t', 'std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()

    for gun in [30, 90]:
        son = g[g['tarih'] >= origin - pd.Timedelta(days=gun)]
        ozet = son.groupby('tanim').agg(**{
            f'son{gun}_log_ort': ('log_t', 'mean'),
            f'son{gun}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')

    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin][['tanim', 'haftanin_gunu', 'tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim', 'haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


def veri_seti_kur(gecmis, hedef, origin, k=30):
    # Eğitim, validation ve test aynı fonksiyonla kurulur. Sadece origin değişir.
    g = gecmis[gecmis['tarih'] < origin].copy()
    g['log_t'] = np.log1p(g['tuketim_duzeltilmis'])
    genel = g['log_t'].mean()

    grup = g.groupby(['il', 'bolge', 'guc_bucket'], observed=True)['log_t'].agg(['sum', 'count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel) / (grup['count'] + k)
    il_guc = g.groupby(['il', 'guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef.copy()
    df = df.merge(grup[['il', 'bolge', 'guc_bucket', 'grup_log_ort']],
                  on=['il', 'bolge', 'guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il', 'guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel)

    df = df.merge(trafo_profili(gecmis, origin), on='tanim', how='left')
    df = df.merge(trafo_gun_profili(gecmis, origin), on=['tanim', 'haftanin_gunu'], how='left')
    df['cold_start'] = df['log_ortalama'].isna()
    return df


def origin_seti(bas, bit, origin):
    hedef = train[(train['tarih'] >= bas) & (train['tarih'] <= bit)]
    return veri_seti_kur(train, hedef, pd.Timestamp(origin))


# Trafo geçmişinden gelen sütunlar. Yeni trafolarda hepsi boş.
PROFIL = [
    'trafo_ortalama', 'trafo_medyan', 'trafo_gun_sayisi',
    'log_ortalama', 'log_medyan', 'log_std', 'sifir_orani',
    'son30_log_ort', 'son90_log_ort', 'son30_sifir', 'son90_sifir',
    'log_trend', 'trafo_gun_orani',
]

# Yeni trafolarda da dolu olan sütunlar (güç, yer, takvim, grup ortalaması).
PROFILSIZ = [
    'guc', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'tatil',
    'il', 'bolge', 'ilce', 'guc_bucket_str', 'grup_log_ort',
]

TAM = PROFILSIZ + PROFIL
print('profilsiz özellik:', len(PROFILSIZ), '| tam özellik (sub_04):', len(TAM))


profilsiz özellik: 11 | tam özellik (sub_04): 24


## Eğitim ve validation

Validation, test ile aynı aylar: Nisan–Temmuz, origin 1 Nisan.
Eğitim hedefleri Mart sonunda bitiyor, sızıntı yok.

Cold-start = origin'den önce hiç görünmeyen trafo. Validationda bu oran
doğal olarak %7.5, testte %22. Mutlak skora değil segmentlere bakıyoruz.


In [ ]:
egitim = pd.concat([
    origin_seti('2025-02-01', '2025-03-31', '2025-02-01'),
    origin_seti('2025-03-01', '2025-03-31', '2025-03-01'),
], ignore_index=True)
val = origin_seti('2025-04-01', '2025-07-31', '2025-04-01')

for df in [egitim, val]:
    df['hedef_log'] = np.log1p(df['tuketim_duzeltilmis'])
    for c in KATEGORIK:
        df[c] = df[c].astype('category')
for c in KATEGORIK:
    val[c] = pd.Categorical(val[c], categories=egitim[c].cat.categories)

print('eğitim', egitim.shape, '| validation', val.shape)
print('eğitim ayları    :', sorted(egitim['ay'].unique()))
print('validation ayları:', sorted(val['ay'].unique()))
print('cold-start oranı -> eğitim:', round(egitim['cold_start'].mean(), 4),
      '| validation:', round(val['cold_start'].mean(), 4))


eğitim (188679, 34) | validation (274929, 34)
eğitim ayları    : [np.int32(2), np.int32(3)]
validation ayları: [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]
cold-start oranı -> eğitim: 0.0134 | validation: 0.075


In [ ]:
def egit(ozellikler, data, val_data=None, agac=3000):
    kat = [c for c in KATEGORIK if c in ozellikler]
    model = lgb.LGBMRegressor(
        n_estimators=agac, learning_rate=0.05, num_leaves=31,
        random_state=42, verbose=-1,
    )
    if val_data is None:
        model.fit(data[ozellikler], data['hedef_log'], categorical_feature=kat)
    else:
        model.fit(
            data[ozellikler], data['hedef_log'],
            eval_X=val_data[ozellikler], eval_y=val_data['hedef_log'],
            eval_metric='rmse', categorical_feature=kat,
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
    return model


def tahmin_al(model, ozellikler, data):
    return np.clip(np.expm1(model.predict(data[ozellikler])), 0, None)


def skor_yaz(ad, tahmin):
    gercek = val['tuketim_duzeltilmis'].to_numpy()
    cs = val['cold_start'].to_numpy()
    toplam = rmsle(gercek, tahmin)
    prof   = rmsle(gercek[~cs], tahmin[~cs])
    cold   = rmsle(gercek[cs], tahmin[cs])
    print(f'{ad:28s}  toplam {toplam:.4f}  |  profilli {prof:.4f}  |  cold-start {cold:.4f}')
    return toplam, prof, cold


## İkili model

İki uzman:

- **A:** sub_04 seti (profil + grup). Geçmişi olan satırlar için.
- **B:** sadece profilsiz özellikler. Yeni trafolar için.

Birleştirme: cold-start ise B, değilse A.


In [ ]:
model_a = egit(TAM, egitim, val)
tahmin_a = tahmin_al(model_a, TAM, val)
skor_yaz('A  tek model (sub_04)', tahmin_a)
print('ağaç:', model_a.best_iteration_)
print()

model_b = egit(PROFILSIZ, egitim, val)
tahmin_b = tahmin_al(model_b, PROFILSIZ, val)
skor_yaz('B  sadece profilsiz', tahmin_b)
print('ağaç:', model_b.best_iteration_)
print()

# B'yi yalnızca cold-start satırlarla da deniyoruz (daha az veri)
egitim_cold = egitim[egitim['cold_start']]
print('B2 eğitim satırı (sadece cold-start):', len(egitim_cold), '/', len(egitim))
model_b2 = egit(PROFILSIZ, egitim_cold, val)
tahmin_b2 = tahmin_al(model_b2, PROFILSIZ, val)
skor_yaz('B2 sadece cold-start veri', tahmin_b2)
print()

cs = val['cold_start'].to_numpy()
skor_yaz('ikili A + B',  np.where(cs, tahmin_b,  tahmin_a))
skor_yaz('ikili A + B2', np.where(cs, tahmin_b2, tahmin_a))
print()
print('Not: ikili modelde profilli skor A ile aynı kalmalı.')


A  tek model (sub_04)         toplam 1.0143  |  profilli 0.8485  |  cold-start 2.1989
ağaç: 58



B  sadece profilsiz           toplam 1.8437  |  profilli 1.8273  |  cold-start 2.0350
ağaç: 81

B2 eğitim satırı (sadece cold-start): 2530 / 188679


B2 sadece cold-start veri     toplam 2.2235  |  profilli 2.2253  |  cold-start 2.2017

ikili A + B                   toplam 0.9883  |  profilli 0.8485  |  cold-start 2.0350
ikili A + B2                  toplam 1.0147  |  profilli 0.8485  |  cold-start 2.2017

Not: ikili modelde profilli skor A ile aynı kalmalı.


## Submission

Profilli satırlara dokunmuyoruz: sub_04 dosyasını taban alıp sadece
cold-start satırları B modelinin tahminiyle değiştiriyoruz.

Böylece Kaggle'daki farkın tamamı yeni trafolardan gelir.
Yerel cold-start skoruna güvenmiyoruz; bu yüzden tek değişkenli bir deneme.


In [ ]:
FINAL_ORIGINLER = [
    ('2025-04-01', '2025-07-31'),
    ('2025-05-01', '2025-08-31'),
    ('2025-06-01', '2025-09-30'),
    ('2025-12-01', '2026-03-31'),
]

final_egitim = pd.concat([origin_seti(b, s, b) for b, s in FINAL_ORIGINLER], ignore_index=True)
final_egitim['hedef_log'] = np.log1p(final_egitim['tuketim_duzeltilmis'])
for c in KATEGORIK:
    final_egitim[c] = final_egitim[c].astype('category')

test_seti = veri_seti_kur(train, test, pd.Timestamp('2026-04-01'))
for c in KATEGORIK:
    test_seti[c] = pd.Categorical(test_seti[c], categories=final_egitim[c].cat.categories)

# Validation küçük sette bulundu, final set daha büyük. Ağaç sayısına pay bırakıyoruz.
agac = int(model_b.best_iteration_ * 1.5)
model_b_final = egit(PROFILSIZ, final_egitim, agac=agac)

taban = pd.read_csv('../submissions/sub_04_kurguA_sade.csv')
sub = test_seti[['id']].merge(taban, on='id', how='left')

cs = test_seti['cold_start'].to_numpy()
yeni = tahmin_al(model_b_final, PROFILSIZ, test_seti)
eski = sub.loc[cs, 'tuketim'].to_numpy()
sub.loc[cs, 'tuketim'] = yeni[cs]

sample = pd.read_csv('../data/raw/sample_submission.csv')
ok = (len(sub) == len(sample)
      and set(sub['id']) == set(sample['id'])
      and sub['tuketim'].isna().sum() == 0
      and (sub['tuketim'] < 0).sum() == 0)
sub.to_csv('../submissions/sub_07_ikili.csv', index=False)

print('final eğitim:', final_egitim.shape)
print('B model ağaç:', agac)
print('değiştirilen satır:', int(cs.sum()), '/', len(sub), f'({cs.mean():.4f})')
print(f'sub_04 cold-start ortalama / medyan: {eski.mean():.1f} / {np.median(eski):.1f}')
print(f'yeni    cold-start ortalama / medyan: {yeni[cs].mean():.1f} / {np.median(yeni[cs]):.1f}')
print(f"doğrulama: {'tamam' if ok else 'HATA'}")
print('kayıt: submissions/sub_07_ikili.csv')


final eğitim: (1306364, 34)
B model ağaç: 121
değiştirilen satır: 158369 / 714688 (0.2216)
sub_04 cold-start ortalama / medyan: 1414.5 / 1333.6
yeni    cold-start ortalama / medyan: 1596.0 / 1254.2
doğrulama: tamam
kayıt: submissions/sub_07_ikili.csv
